# [7.5] Predictive Concept Decoders - Exercises

Build the local PCD contract: question-labeled batches, sparse concept encoders, question-conditioned decoders, baseline comparison, stability checks, and top-vs-low-margin active removal controls.

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part5_predictive_concept_decoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_predictive_concept_decoders.tests as tests

GT_TIER = "GT-3"
EXERCISE_ID = "7.5.predictive_concept_decoders"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1 minute for CUDA preflight"
REQUIRES_GPU = False

## Behavioral Question Batches

Keep activations aligned with question ids, answer ids, and the readable PCD question bank.

In [ ]:
@dataclass(frozen=True)
class PCDQuestionBatch:
    activations: t.Tensor
    question_ids: t.Tensor
    answer_ids: t.Tensor
    question_texts: tuple[str, ...]


def default_pcd_questions() -> tuple[str, ...]:
    raise NotImplementedError()


def build_pcd_question_batch(
    activations: t.Tensor,
    question_ids: t.Tensor,
    answer_ids: t.Tensor,
    question_texts: tuple[str, ...] | None = None,
) -> PCDQuestionBatch:
    raise NotImplementedError()


tests.test_build_pcd_question_batch_validates_shapes_and_questions(
    build_pcd_question_batch,
    default_pcd_questions,
)

## Sparse Concept Encoding

Project activations onto concept directions, apply ReLU and thresholding, optionally keep top-k concepts, then report density.

In [ ]:
@dataclass(frozen=True)
class ConceptSparsityReport:
    mean_l0: float
    density: float
    passes_sparsity: bool


def sparse_concept_encode(
    activations: t.Tensor,
    concept_directions: t.Tensor,
    *,
    bias: t.Tensor | None = None,
    top_k: int | None = None,
    threshold: float = 0.0,
) -> t.Tensor:
    raise NotImplementedError()


def concept_sparsity_report(
    concepts: t.Tensor,
    *,
    active_threshold: float = 0.0,
    max_density: float = 0.3,
) -> ConceptSparsityReport:
    raise NotImplementedError()


tests.test_sparse_concept_encode_and_sparsity_controls(
    sparse_concept_encode,
    concept_sparsity_report,
)

## Question-Conditioned Decoding

Answer logits should depend on both the sparse concept vector and the behavioral question embedding. In the full path, you will train a tiny decoder over explicit concept-question interaction features; a question-agnostic probe should fail on rows where the same activation is asked opposite behavioral questions.

In [ ]:
def question_conditioned_decoder_logits(
    concepts: t.Tensor,
    question_embeddings: t.Tensor,
    decoder_weight: t.Tensor,
    *,
    decoder_bias: t.Tensor | None = None,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_question_conditioned_decoder_uses_question_information(
    question_conditioned_decoder_logits,
)

In [ ]:
@dataclass(frozen=True)
class PCDComparisonReport:
    pcd_accuracy: float
    probe_accuracy: float
    sae_classifier_accuracy: float
    activation_oracle_accuracy: float
    best_baseline_accuracy: float
    beats_probe: bool
    beats_best_baseline: bool


def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def pcd_comparison_report(
    pcd_logits: t.Tensor,
    probe_logits: t.Tensor,
    sae_classifier_logits: t.Tensor,
    activation_oracle_logits: t.Tensor,
    answer_ids: t.Tensor,
) -> PCDComparisonReport:
    raise NotImplementedError()


tests.test_pcd_comparison_report_beats_baselines(pcd_comparison_report)

## Concept Audit Controls

Stable, useful concepts should survive seed changes, matter more than low-margin active-control removal, and have legible names. Treat the names as an audit aid, not proof that the concept is semantically pure.

In [ ]:
@dataclass(frozen=True)
class ConceptStabilityReport:
    top_concepts_by_seed: tuple[tuple[int, ...], ...]
    mean_pairwise_jaccard: float
    stable: bool


@dataclass(frozen=True)
class ConceptRemovalReport:
    original_answer: int
    top_removed_answer: int
    random_removed_answer: int
    top_removal_changed: bool
    random_removal_changed: bool
    top_removal_delta: float
    random_removal_delta: float
    random_removal_does_less: bool


@dataclass(frozen=True)
class ConceptAuditReport:
    selected_concept_ids: tuple[int, ...]
    selected_concept_names: tuple[str, ...]
    explanation: str
    names_expected_cluster: bool


def concept_stability_report(
    concept_scores_by_seed: list[t.Tensor],
    *,
    top_k: int = 3,
    min_jaccard: float = 0.5,
) -> ConceptStabilityReport:
    raise NotImplementedError()


def concept_removal_report(
    original_logits: t.Tensor,
    top_removed_logits: t.Tensor,
    random_removed_logits: t.Tensor,
    *,
    target_answer_id: int | None = None,
) -> ConceptRemovalReport:
    raise NotImplementedError()


def concept_audit_report(
    concept_scores: t.Tensor,
    concept_names: list[str],
    expected_cluster_terms: list[str],
    *,
    top_k: int = 2,
) -> ConceptAuditReport:
    raise NotImplementedError()


tests.test_concept_stability_removal_and_audit_controls(
    concept_stability_report,
    concept_removal_report,
    concept_audit_report,
)

## Combined Contract

After each helper passes, compose a CPU-only report with every local metric and control. Then inspect the committed CUDA report: it should use four behavioral questions, train a question-conditioned decoder, keep trained non-interaction baselines at chance, degrade under question shuffling, and make top active concept removal more damaging than low-margin active-control removal.

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
